# DQAOA-GPT HUBO Pipeline

This notebook runs the full pipeline:
1. **Step 1** – Generate HUBO ADAPT-QAOA training data
2. **Step 2** – Tokenize the data for nanoGPT
3. **Step 3** – Train nanoGPT on the tokenized data
4. **Step 4** – Solve a large HUBO with DQAOA (GPT or standard QAOA)

Edit the `# --- Parameters ---` blocks in each step before running.

In [ ]:
import os
import sys
import time
import shutil
import subprocess

import numpy as np

# Make sure project root is on the path
NOTEBOOK_DIR = os.path.abspath('.')
sys.path.insert(0, NOTEBOOK_DIR)

print(f'Working directory : {NOTEBOOK_DIR}')

---
## Step 1 – Generate HUBO ADAPT-QAOA Training Data

In [ ]:
from qaoa_gpt_src.generate_adapt_hubo_data import generate_data_hubo

# --- Parameters ---
n_nodes          = 4
hubo_number      = 500
include_linear   = True
include_quadratic= True
include_cubic    = True
density          = 0.5
weight_range     = 1.0
rounding_digits  = 1      # 1 → smaller vocab (recommended)
use_brute_force  = True
use_sa           = True
op_pool          = 'all_pool'  # 'all_pool' | 'qaoa_mixer' | 'qaoa_single_x' | 'qaoa_double_ops'
optimizer        = 'BFGS'      # 'BFGS' | 'L-BFGS-B' | 'COBYLA'
approx_ratio     = 0.97
max_iter         = 10
trials           = 1
init_gamma       = [0.01]
multi_gamma      = False
from_formulated  = False
hubo_size_s1     = None   # only used when from_formulated=True
npy_dir          = None   # path to pre-saved .npy HUBOs, or None
output_dir_s1    = 'adapt_results_hubo'
seed_hubo_s1     = 42
seed_adapt       = None
verbose_s1       = True
# ------------------

print('=' * 60)
print('STEP 1: Generating HUBO ADAPT-QAOA training data')
print('=' * 60)
print(f'  n_nodes        : {n_nodes}')
print(f'  hubo_number    : {hubo_number}')
print(f'  weight_range   : {weight_range}')
print(f'  density        : {density}')
print(f'  rounding_digits: {rounding_digits}')
print(f'  op_pool        : {op_pool}')
print(f'  optimizer      : {optimizer}')
print(f'  approx_ratio   : {approx_ratio}')
print(f'  max_iter       : {max_iter}')
print(f'  output_dir     : {output_dir_s1}')
print('=' * 60)

generate_data_hubo(
    output_dir=output_dir_s1,
    hubo_tensors=None,
    hubo_input_npy_dir=npy_dir,
    from_formulated=from_formulated,
    hubo_size=hubo_size_s1,
    n_nodes=n_nodes,
    hubo_number=hubo_number,
    include_linear=include_linear,
    include_quadratic=include_quadratic,
    include_cubic=include_cubic,
    density=density,
    weight_range=weight_range,
    rounding_digits=rounding_digits,
    use_brute_force=use_brute_force,
    use_simulated_annealing=use_sa,
    op_pool=op_pool,
    init_gamma=init_gamma,
    trials_per_hubo=trials,
    optimizer=optimizer,
    approx_ratio=approx_ratio,
    max_iter=max_iter,
    multi_gamma=multi_gamma,
    seed_hubo=seed_hubo_s1,
    seed_adapt=seed_adapt,
    verbose=verbose_s1,
)

print(f'\nResults saved to: {output_dir_s1}/res/  and  {output_dir_s1}/hubo/')

---
## Step 2 – Tokenize HUBO Data for nanoGPT

In [ ]:
from qaoa_gpt_src.prepare_tokens_hubo import generate_tokens_hubo

# --- Parameters ---
results_dir          = 'adapt_results_hubo'    # output_dir from step 1
extra_results_dirs   = ''                      # semicolon-separated extra dirs, or ''
save_dir             = 'nanoGPT/data/hubo_save_dir'
config_template_dir  = 'qaoa_gpt_src'
n_nodes_s2           = 4                       # must match step 1
rounding_digits_s2   = 1
min_block_size       = 128
max_block_size       = 256
max_abs_param        = 10.0
perform_coef_mod     = True
apply_sliding_window = True
skip_only_qaoa       = False
val_frac             = 0.1
test_frac            = 0.1
approx_ratio_thr     = 0.5
n_workers            = 1
debug_limit          = 0
verbose_s2           = True
# ------------------

results_path = results_dir
if extra_results_dirs:
    results_path += ';' + extra_results_dirs

print('=' * 60)
print('STEP 2: Tokenizing HUBO data for nanoGPT')
print('=' * 60)
print(f'  results_dir   : {results_path}')
print(f'  save_dir      : {save_dir}')
print(f'  n_nodes       : {n_nodes_s2}')
print(f'  max_block_size: {max_block_size}')
print(f'  val_frac      : {val_frac}')
print(f'  test_frac     : {test_frac}')
print(f'  approx_thr    : {approx_ratio_thr}')
print('=' * 60)

generate_tokens_hubo(
    results_fpath_str=results_path,
    save_path_str=save_dir,
    config_path_temp=config_template_dir,
    n_nodes=n_nodes_s2,
    rounding_digits=rounding_digits_s2,
    min_block_size=min_block_size,
    max_block_size=max_block_size,
    val_frac=val_frac,
    test_frac=test_frac,
    max_abs_param_val=max_abs_param,
    perform_coef_mod_range=perform_coef_mod,
    apply_sliding_window=apply_sliding_window,
    n_workers=n_workers,
    skip_only_qaoa_circ=skip_only_qaoa,
    approx_ratio_thr=approx_ratio_thr,
    debug_limit=debug_limit,
    verbose=verbose_s2,
)

print(f'\nTokenized data saved to: {save_dir}/')

---
## Step 3 – Train nanoGPT on HUBO Data

In [ ]:
# --- Parameters ---
data_dir_s3    = 'nanoGPT/data/hubo_save_dir'  # output of step 2
nanogpt_dir    = 'nanoGPT'
train_script   = 'train_pad_gemb.py'
# Set to None to use config defaults
max_iters      = None
batch_size     = None
n_layer        = None
n_head         = None
n_embd         = None
dropout        = None
learning_rate  = None
device_s3      = None     # 'cuda' | 'cpu' | 'mps' | None
compile_model  = None     # 'true' | 'false' | None
eval_interval  = None
ckpt_name      = None     # e.g. 'ckpt_4nodes.pt', or None
# ------------------

nanogpt_abs    = os.path.join(NOTEBOOK_DIR, nanogpt_dir)
train_script_path = os.path.join(nanogpt_abs, train_script)
config_file    = os.path.join(NOTEBOOK_DIR, data_dir_s3, 'train_adapt_gpt_config.py')

assert os.path.isfile(train_script_path), f'Training script not found: {train_script_path}'
assert os.path.isfile(config_file), (
    f'Config not found: {config_file}\n'
    'Did you run Step 2 first?'
)

cmd = [sys.executable, train_script_path, config_file]

overrides = {
    'max_iters':     max_iters,
    'batch_size':    batch_size,
    'n_layer':       n_layer,
    'n_head':        n_head,
    'n_embd':        n_embd,
    'dropout':       dropout,
    'learning_rate': learning_rate,
    'device':        f"'{device_s3}'" if device_s3 else None,
    'compile':       compile_model,
    'eval_interval': eval_interval,
}
for key, val in overrides.items():
    if val is not None:
        cmd.append(f'--{key}={val}')

print('=' * 60)
print('STEP 3: Training nanoGPT on HUBO data')
print('=' * 60)
print(f'  Script : {train_script_path}')
print(f'  Config : {config_file}')
active_overrides = {k: v for k, v in overrides.items() if v is not None}
if active_overrides:
    print('  Overrides:')
    for k, v in active_overrides.items():
        print(f'    {k} = {v}')
print('=' * 60)
print(f'\nRunning: {" ".join(cmd)}\n')

result = subprocess.run(cmd, cwd=nanogpt_abs)

if result.returncode == 0:
    print('\nStep 3 complete.')
    data_stem = os.path.basename(data_dir_s3.rstrip('/'))
    out_dir   = os.path.join(nanogpt_abs, f'out-{data_stem}')

    for fname in ['meta.pkl', 'train_adapt_gpt_config.py']:
        src = os.path.join(NOTEBOOK_DIR, data_dir_s3, fname)
        dst = os.path.join(out_dir, fname)
        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f'  Copied {fname} → {out_dir}/')

    if ckpt_name:
        name = ckpt_name if ckpt_name.endswith('.pt') else ckpt_name + '.pt'
        best_ckpt  = os.path.join(out_dir, 'ckpt.pt')
        named_ckpt = os.path.join(out_dir, name)
        if os.path.exists(best_ckpt):
            shutil.copy2(best_ckpt, named_ckpt)
            print(f'  Best checkpoint → {named_ckpt}')
        else:
            print(f'  WARNING: ckpt.pt not found in {out_dir}; skipping rename.')

    print(f'Checkpoint saved to: {out_dir}/')
else:
    raise RuntimeError(f'Training exited with code {result.returncode}')

---
## Step 4 – Solve Large HUBO with DQAOA

When loading from `txt` or `formulated` mode, the factorized form `(Q, k_third)` is
automatically used for energy evaluation, reducing the per-iteration cost from
**O(N³)** to **O(N²)**. For `random` or `npy` mode the dense evaluator is used.

In [ ]:
from qaoa_gpt_src.hubo_hamiltonian import generate_random_hubo
from subHUBO_gen import formulate_HUBO, load_QUBO, load_k_third
from DQAOA import DQAOA

# --- Parameters ---
solver_s4        = 'gpt'      # 'gpt' | 'qaoa'

# [gpt] paths – set these if solver_s4 == 'gpt'
ckpt_path        = None       # e.g. 'nanoGPT/out-hubo_save_dir/ckpt.pt'
data_dir_s4      = None       # defaults to dirname(ckpt_path)
config_file_s4   = None       # defaults to <data_dir_s4>/train_adapt_gpt_config.py
device_s4        = 'cuda'     # 'cuda' | 'cpu'
pool_type        = 'all_pool'

# [qaoa] settings
qaoa_num_layers  = 2
qaoa_max_iters   = 200
qaoa_target      = 'nvidia'   # 'nvidia' | 'qpp-cpu'
qaoa_reverse_bitorder = False

# HUBO source – choose one mode:
#   'random'     : generate a random HUBO  (dense O(N^3) energy eval)
#   'npy'        : load from a .npy file   (dense O(N^3) energy eval)
#   'formulated' : load via formulate_HUBO (factorized O(N^2) energy eval)
#   'txt'        : load from QUBO/k_third text files (factorized O(N^2) energy eval)
hubo_source      = 'txt'
hubo_npy_path    = None       # used when hubo_source == 'npy'
hubo_size_s4     = 100
ctn              = 0          # copy number for 'txt' mode
hubo_dir         = './HUBOs'
density_s4       = 0.5
weight_range_s4  = 1.0
seed_hubo_s4     = 0

# DQAOA loop settings
sub_n            = 4          # must match n_nodes from training
iterations       = 200
num_shots        = 1024
seed_dq          = 12345
save_solution    = None       # path to save solution .npy, or None
verbose_s4       = False
# ------------------

# Resolve gpt paths
ckpt = data_dir_resolved = config_file_resolved = None
if solver_s4 == 'gpt':
    assert ckpt_path is not None, 'Set ckpt_path for solver_s4="gpt"'
    ckpt                 = ckpt_path
    data_dir_resolved    = data_dir_s4 or os.path.dirname(os.path.abspath(ckpt))
    config_file_resolved = config_file_s4 or os.path.join(data_dir_resolved,
                                                           'train_adapt_gpt_config.py')
    for path, label in [(ckpt, 'checkpoint'),
                        (data_dir_resolved, 'data_dir'),
                        (config_file_resolved, 'config_file')]:
        assert os.path.exists(path), f'{label} not found: {path}'

# Load / generate HUBO tensor
Q_full = k_third_full = None   # factorized form; populated for txt/formulated modes

if hubo_source == 'npy':
    print(f'Loading HUBO tensor from: {hubo_npy_path}')
    T_full    = np.load(hubo_npy_path)
    hubo_size = T_full.shape[0]

elif hubo_source == 'formulated':
    print(f'Loading formulated HUBO (size={hubo_size_s4})...')
    T_full       = formulate_HUBO(hubo_size_s4)
    Q_full       = load_QUBO(hubo_size_s4)
    k_third_full = load_k_third(hubo_size_s4)
    hubo_size    = hubo_size_s4

elif hubo_source == 'txt':
    print(f'Loading HUBO from text files (size={hubo_size_s4}, ctn={ctn})...')
    def _find_file(directory, size, suffix, ctn):
        for name in [f'{size}_{suffix}_Copy{ctn}.txt',
                     f'{size}_{suffix}-Copy{ctn}.txt',
                     f'{size}_{suffix}.txt']:
            p = os.path.join(directory, name)
            if os.path.exists(p): return p
        raise FileNotFoundError(f'Cannot find {suffix} file.')

    q_path  = _find_file(hubo_dir, hubo_size_s4, 'QUBO',    ctn)
    kt_path = _find_file(hubo_dir, hubo_size_s4, 'k_third', ctn)
    print(f'  Q matrix : {q_path}')
    print(f'  k_third  : {kt_path}')

    with open(q_path) as f:
        Q_full = np.array(np.reshape(np.asmatrix(f.read()), (hubo_size_s4, hubo_size_s4)))
    with open(kt_path) as f:
        k_third_full = np.array(np.reshape(np.asmatrix(f.read()), (hubo_size_s4, 8)))

    T_full = np.zeros((hubo_size_s4, hubo_size_s4, hubo_size_s4))
    for i in range(hubo_size_s4):
        T_full[i, i, i] = Q_full[i, i]
    for i in range(hubo_size_s4):
        for j in range(i + 1, hubo_size_s4):
            T_full[i, i, j] = Q_full[i, j]
    for i in range(hubo_size_s4):
        for j in range(i + 1, hubo_size_s4):
            for k in range(j + 1, hubo_size_s4):
                T_full[i, j, k] = float(np.sum(
                    k_third_full[i, :] * k_third_full[j, :] * k_third_full[k, :]))
    hubo_size = hubo_size_s4

else:  # 'random'
    print(f'Generating random HUBO (size={hubo_size_s4}, seed={seed_hubo_s4})...')
    T_full    = generate_random_hubo(hubo_size_s4,
                                     density=density_s4,
                                     weight_range=weight_range_s4,
                                     seed=seed_hubo_s4)
    hubo_size = hubo_size_s4

energy_mode = 'factorized O(N^2)' if Q_full is not None else 'dense O(N^3)'

print('=' * 60)
print('STEP 4: Solving large HUBO with DQAOA')
print('=' * 60)
print(f'  solver      : {solver_s4}')
print(f'  HUBO size   : {hubo_size}')
print(f'  sub_n       : {sub_n}')
print(f'  iterations  : {iterations}')
print(f'  energy eval : {energy_mode}')
if solver_s4 == 'gpt':
    print(f'  checkpoint  : {ckpt}')
    print(f'  device      : {device_s4}')
else:
    print(f'  num_layers  : {qaoa_num_layers}')
    print(f'  max_iters   : {qaoa_max_iters}  (COBYLA per sub-HUBO)')
    print(f'  target      : {qaoa_target}')
print('=' * 60)

print('\nStarting DQAOA loop...')
dqaoa = DQAOA(
    T=T_full,
    sub_n=sub_n,
    iterations=iterations,
    model_ckpt=ckpt,
    data_dir=data_dir_resolved,
    config_file=config_file_resolved,
    device=device_s4,
    pool_type=pool_type,
    num_shots=num_shots,
    seed=seed_dq,
    verbose=verbose_s4,
    qaoa_num_layers=qaoa_num_layers,
    qaoa_max_iters=qaoa_max_iters,
    qaoa_target=qaoa_target,
    qaoa_reverse_bitorder=qaoa_reverse_bitorder,
    solver=solver_s4,
    Q=Q_full,
    k_third=k_third_full,
)

t0 = time.time()
x_opt, best_energy = dqaoa.run()
elapsed = time.time() - t0

print('\n' + '=' * 60)
print('Results')
print('=' * 60)
print(f'  Best HUBO energy  : {best_energy:.6f}')
print(f'  Total time        : {elapsed:.6f}s')
print(f'  Solution (first 20 vars): {x_opt[:20]}')

if save_solution:
    np.save(save_solution, x_opt)
    print(f'\nSolution saved to: {save_solution}')